In [1]:
import pyspark, os, sys, random, pandas as pd
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
import pyspark.sql.types as T
from operator import add
os.environ['PYSPARK_PYTHON'] = os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable
print("Python", sys.executable)
print("Java", os.environ["JAVA_HOME"])
spark = SparkSession.builder.appName("spark-cards").getOrCreate()
sc = spark.sparkContext
print("Spark version:", spark.version)
spark

Python /usr/local/bin/python
Java /usr/lib/jvm/java-17-openjdk-amd64


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/08/19 20:01:54 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 4.1.2


References:
- https://spark.apache.org/docs/latest/rdd-programming-guide.html
- https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.RDD.html

## Building the deck

In [2]:
SUITS = ["CLUBS", "DIAMONDS", "HEARTS", "SPADES"]
RANKS = ["ACE", "TWO", "THREE", "FOUR", "FIVE", "SIX", "SEVEN", "EIGHT",
         "NINE", "TEN", "JACK", "QUEEN", "KING"]

def make_deck(joker=False):
    """Return a deck as a list of dicts"""
    deck = [{"suit": s, "rank": r} for s in SUITS for r in RANKS]
    if joker:
        deck.append({"suit": "JOKER", "rank": "JOKER"})
    return deck

def card_str(card):                       # equivalent of Java's Card.toString()
    return f"{card['rank']} of {card['suit']}"

cards = make_deck(joker=True)
random.shuffle(cards)                     # Collections.shuffle(cards)

print(len(cards), "cards")
print([card_str(c) for c in cards[:5]])

53 cards
['NINE of DIAMONDS', 'ACE of HEARTS', 'FOUR of DIAMONDS', 'JACK of HEARTS', 'JOKER of JOKER']


## Creating the RDD

`sc.parallelize` splits the local list into partitions. With no explicit number, Spark
uses `spark.default.parallelism` -- locally that is the number of cores, which is why the
Java version printed a number that depends on the machine (`local[*]`).

In [3]:
card_rdd = sc.parallelize(cards)
print("Partitions:", card_rdd.getNumPartitions())
print("Count:", card_rdd.count())          # count() is an ACTION -- it runs the job

Partitions: 32


[Stage 0:>                                                        (0 + 32) / 32]

Count: 53


## Query 1: find the aces

`filter` is a **transformation**: nothing happens until `collect` (an **action**) pulls
the results back to the driver.

Careful with `collect()` -- it materialises the whole result in driver memory. Fine for 4
aces, dangerous for a real dataset; use `take(n)` or write to storage instead.

In [4]:
aces = card_rdd.filter(lambda c: c["rank"] == "ACE").collect()
print([card_str(c) for c in aces])

['ACE of HEARTS', 'ACE of DIAMONDS', 'ACE of CLUBS', 'ACE of SPADES']


## Query 2: sort by suit, then find the hearts

The Java line was:

```java
var cardRdd2 = cardRdd.sortBy(card -> card.getSuit(), true, 5);
```

Two differences in Python:

1. The extra arguments are keyword arguments: `ascending=True, numPartitions=5`.
2. Java enums sort in **declaration order**. Our suits are plain strings, so sorting them
   directly would give alphabetical order (CLUBS, DIAMONDS, HEARTS, JOKER, SPADES). To
   keep the Java ordering we sort by the position of the suit in a small lookup dict.

That dict is captured by the lambda, so Spark ships a copy of it to every executor -- the
same rule as Java's `Serializable`, just automatic.

`sortBy` performs a **range partitioning shuffle**: all clubs land in the first
partitions, the joker in the last. This is where data moves across the network.

In [5]:

card_rdd2 = card_rdd.sortBy(lambda c: c["suit"],
                            ascending=True, numPartitions=5)
print("Partitions:", card_rdd2.getNumPartitions())

hearts = card_rdd2.filter(lambda c: c["suit"] == "HEARTS").collect()
print(len(hearts), "hearts:", [card_str(c) for c in hearts])

Partitions: 5
13 hearts: ['ACE of HEARTS', 'JACK of HEARTS', 'TEN of HEARTS', 'FIVE of HEARTS', 'EIGHT of HEARTS', 'TWO of HEARTS', 'SEVEN of HEARTS', 'QUEEN of HEARTS', 'SIX of HEARTS', 'NINE of HEARTS', 'THREE of HEARTS', 'FOUR of HEARTS', 'KING of HEARTS']


### Looking inside the partitions

`glom()` turns each partition into a list, which is a handy way to *see* how the data was
distributed. Compare the unsorted RDD (suits mixed everywhere) with the sorted one
(suits grouped, though partitions are rarely of equal size).

In [6]:
def partition_summary(rdd, label):
    print(label)
    for i, part in enumerate(rdd.glom().collect()):
        suits = sorted({c["suit"] for c in part})
        print(f"  partition {i}: {len(part):2d} cards  {suits}")

partition_summary(card_rdd, "Original RDD")
partition_summary(card_rdd2, "After sortBy(suit)")

Original RDD
  partition 0:  1 cards  ['DIAMONDS']
  partition 1:  2 cards  ['DIAMONDS', 'HEARTS']
  partition 2:  1 cards  ['HEARTS']
  partition 3:  2 cards  ['DIAMONDS', 'JOKER']
  partition 4:  2 cards  ['DIAMONDS', 'HEARTS']
  partition 5:  1 cards  ['HEARTS']
  partition 6:  2 cards  ['DIAMONDS']
  partition 7:  2 cards  ['CLUBS', 'DIAMONDS']
  partition 8:  1 cards  ['SPADES']
  partition 9:  2 cards  ['HEARTS', 'SPADES']
  partition 10:  2 cards  ['DIAMONDS', 'HEARTS']
  partition 11:  1 cards  ['CLUBS']
  partition 12:  2 cards  ['DIAMONDS', 'SPADES']
  partition 13:  2 cards  ['DIAMONDS', 'HEARTS']
  partition 14:  1 cards  ['HEARTS']
  partition 15:  2 cards  ['DIAMONDS']
  partition 16:  2 cards  ['CLUBS']
  partition 17:  1 cards  ['SPADES']
  partition 18:  2 cards  ['HEARTS', 'SPADES']
  partition 19:  2 cards  ['CLUBS', 'HEARTS']
  partition 20:  1 cards  ['SPADES']
  partition 21:  2 cards  ['CLUBS']
  partition 22:  2 cards  ['CLUBS', 'HEARTS']
  partition 23:  1 card

## Query 3: find the joker

The Java file left a comment here -- *"try with cardRdd2"*. The point is that the answer
is identical: a shuffle **reorganises** records across partitions, it never drops them.
What changes is only *where* the joker ends up. Since `JOKER` is last in `SUIT_ORDER`, it
lands in the final partition of `card_rdd2`.

In [7]:
jokers  = card_rdd.filter(lambda c: c["suit"] == "JOKER").collect()
jokers2 = card_rdd2.filter(lambda c: c["suit"] == "JOKER").collect()
print("unsorted:", jokers)
print("sorted:  ", jokers2)

# which partition holds the joker after sorting?
idx = card_rdd2.mapPartitionsWithIndex(
    lambda i, it: [(i, card_str(c)) for c in it if c["suit"] == "JOKER"]).collect()
print("found in partition:", idx)

unsorted: [{'suit': 'JOKER', 'rank': 'JOKER'}]
sorted:   [{'suit': 'JOKER', 'rank': 'JOKER'}]
found in partition: [(3, 'JOKER of JOKER')]


## Bonus: counting cards per suit (MapReduce)

The same `map` -> `reduceByKey` pattern as the word count example, applied to cards:
each card becomes a `(suit, 1)` pair, and `reduceByKey` sums them per key.

Note that the pairs are **tuples**, not dicts -- Spark reads the key from position 0 and
the value from position 1, so any keyed operation (`reduceByKey`, `groupByKey`, `join`)
expects a 2-tuple.

`toDebugString()` shows the `ShuffledRDD` boundary -- the reduce phase.

In [8]:
counts = (card_rdd
    .map(lambda c: (c["suit"], 1))
    .reduceByKey(add))

print(counts.toDebugString().decode())
print()
for suit, n in sorted(counts.collect(), key=lambda x: -x[1]):
    print(f"{suit:10s} {n}")

(32) PythonRDD[19] at RDD at PythonRDD.scala:58 []
 |   MapPartitionsRDD[18] at mapPartitions at PythonRDD.scala:170 []
 |   ShuffledRDD[17] at partitionBy at NativeMethodAccessorImpl.java:0 []
 +-(32) PairwiseRDD[16] at reduceByKey at /tmp/ipykernel_1662/1464152222.py:3 []
    |   PythonRDD[15] at reduceByKey at /tmp/ipykernel_1662/1464152222.py:3 []
    |   ParallelCollectionRDD[0] at readRDDFromFile at PythonRDD.scala:299 []

SPADES     13
HEARTS     13
CLUBS      13
DIAMONDS   13
JOKER      1


# Shutdown Spark when done

In [9]:
spark.stop()